In [103]:
import numpy as np
import transforms3d as td
import pandas as pd
import bokeh.io
import bokeh.plotting
# Import libraries
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.signal import peak_widths, find_peaks, peak_prominences, medfilt
import datetime
from sklearn.decomposition import PCA
import re
import panel as pn
from bokeh.models import ColumnDataSource
from bokeh.palettes import Viridis
bokeh.io.output_notebook()
pn.extension()
import plotly.graph_objects as go
from bokeh.models import HoverTool 
from bokeh.io import export_png

Loading BokehJS ...

In [105]:
class CSVParser(object):
    def __init__(self):
        self.conditions = []
        self.calibration_conditions = []
        self.df = None
        self.calibration_df = None
        self.calibration_dict = None 
        self.angle_plots = {}
        
    def file_to_df(self, filename):
        df = pd.read_csv(filename, low_memory=False)
        df.drop(columns = ['Unnamed: 0'], inplace = True)
        df.sort_values(by=['time'], inplace = True, ignore_index = True)
        df['trakstar0'] = df['trakstar0'].apply(self.parse_transform)
        df['trakstar1'] = df['trakstar1'].apply(self.parse_transform)
        df['trakstar2'] = df['trakstar2'].apply(self.parse_transform)
        df['trakstar_01'] = [np.dot(np.linalg.inv(t1), t2) for t1, t2 in zip(df['trakstar0'], df['trakstar1'])]
        df['trakstar_02'] = [np.dot(np.linalg.inv(t1), t2) for t1, t2 in zip(df['trakstar0'], df['trakstar2'])]
        df["mcp_angle"] = np.empty(len(df.index))
        df["pip_angle"] = np.empty(len(df.index))
        df["finger_angle"] = np.empty(len(df.index))
        df['motor_position'] = df['motor_position'] * 0.007 # encoder to mm
        self.df = df

    def plot_data_pca_plane(self, calibration):

        sub_df = self.df[self.df['condition'] == calibration].copy()
        centered_points = sub_df[['trakstar_02_x', 'trakstar_02_y', 'trakstar_02_z']] - sub_df[['trakstar_02_x', 'trakstar_02_y', 'trakstar_02_z']].mean()
        normal_vector = self.calibration_df[self.calibration_df['condition'] == calibration]['PCA_axis3'].iloc[0]
        pca_axis1 = self.calibration_df[self.calibration_df['condition'] == calibration]['PCA_axis1'].iloc[0]
        pca_axis2 = self.calibration_df[self.calibration_df['condition'] == calibration]['PCA_axis2'].iloc[0]

        # Create a 3D scatter plot for the centered points
        fig = go.Figure()
        fig.add_trace(go.Scatter3d(x=centered_points['trakstar_02_x'], y=centered_points['trakstar_02_y'], z=centered_points['trakstar_02_z'],
                                mode='markers', name='Centered 3D Points'))

        # Plot the normal vector
        fig.add_trace(go.Scatter3d(x=[0, normal_vector[0]], y=[0, normal_vector[1]], z=[0, normal_vector[2]],
                                mode='lines+markers', name='Normal Vector'))
        # Plot the normal vector
        fig.add_trace(go.Scatter3d(x=[-pca_axis1[0], pca_axis1[0]], y=[-pca_axis1[1], pca_axis1[1]], z=[-pca_axis1[2], pca_axis1[2]],
                                mode='lines+markers', name='PCA axis 1'))
        # Plot the normal vector
        fig.add_trace(go.Scatter3d(x=[-pca_axis2[0], pca_axis2[0]], y=[-pca_axis2[1], pca_axis2[1]], z=[-pca_axis2[2], pca_axis2[2]],
                                mode='lines+markers', name='PCA axis 2'))

        # Plot the best-fitting plane
        #x_plane, y_plane, z_plane = np.meshgrid(np.linspace(-pca_axis1[0], pca_axis1[0], 10),
                                        #np.linspace(-pca_axis2[0], pca_axis2[0], 10))

        #fig.add_trace(go.Surface(x=x_plane, y=y_plane, z=z_plane, opacity=0.5, colorscale='Viridis', showscale=False, name='Best-Fitting Plane'))

        # Define the axis range
        axis_range = [-0.25, 0.25]

        # Update layout with explicit axis range
        fig.update_layout(scene=dict(aspectmode="cube",
                                    xaxis=dict(range=axis_range),
                                    yaxis=dict(range=axis_range),
                                    zaxis=dict(range=axis_range)),
            width = 1000,
            height = 800
        )
        # Show the plot
        fig.show()

    def parse_conditions(self):
        '''
        Get all conditions from dataset.
        '''
        conditions = list(self.df['condition'].unique())
        for condition in conditions:
            if 'calibration' in condition:
                self.calibration_conditions.append(condition)

        self.conditions = set(conditions) - set(self.calibration_conditions)

    def parse_transform(self, transform_str):
        # Split the string using regex
        result = re.split(r'[:\n]', transform_str)

        # Remove empty strings from the result
        result = [item.strip() for item in result if item.strip()]
        x = float(result[2])
        y = float(result[4])
        z = float(result[6])
        rx = float(result[9])
        ry = float(result[11])
        rz = float(result[13])
        rw = float(result[15])
        # Display the result
        return td.affines.compose([x,y,z], td.quaternions.quat2mat([rw, rx, ry, rz]), np.ones(3))

    def relative_transform(self, t1, t2):
        return np.dot(np.linalg.inv(t1), t2)

    def get_calibration_pca(self, calibration):
        '''
        input: dataframe & column for calibration
        '''
        data = self.df[self.df['condition'] == calibration]['trakstar_02']
        n = len(data)
        points = np.zeros((n, 3))

        for i, transform in enumerate(data):
            translation = transform[:3, 3]
            points[i, :3] = translation

        pca = PCA()
        pca.fit(points)

        return pca

    def twist_rotation_about_axis(self, transform, axis):
        '''
        Returns the angle theta in degrees about the twist axis. 
        '''
        # quaternion convention is [w, x, y, z]
        transform_quat = td.quaternions.mat2quat(transform[:3, :3])
        transform_axes = np.array([transform_quat[1], transform_quat[2], transform_quat[3]])
        
        dot_product = np.dot(transform_axes, axis)
        dp_sign = np.sign(dot_product)
        axis_norm = np.linalg.norm(axis)
        projection = (dot_product / axis_norm**2) * axis 
        
        # define twist in quaternion form and normalize
        twist = np.array([transform_quat[0], projection[0], projection[1], projection[2]])
        twist /= td.quaternions.qnorm(twist)
        
        # catching singularities
        threshold = 1e-6
        if td.quaternions.qnorm(twist) < threshold:
            print("Singularity in twist")
            return
        elif td.quaternions.qnorm(transform_quat) < threshold:
            print("Singularity in rotation")
            return
        
        twist = dp_sign * twist
        twist_axis, twist_theta = td.quaternions.quat2axangle(twist)

        if not np.allclose(axis, twist_axis):
            print("Axis of rotation is not the given axis. Something went wrong.")
            print("twist axis: %s"%(twist_axis))
            print("pca axis  : %s"%(axis))
        
        return dp_sign*twist_theta*(180/np.pi)

    def calculate_relative_angle(self, ref_transform, target_transform, axis):
        
        net_transform = np.dot(np.linalg.inv(ref_transform), target_transform)
        target_theta = self.twist_rotation_about_axis(net_transform, axis)
        theta_sign = np.sign(target_theta)
        delta = abs(target_theta)
        theta = min(delta, 360 - delta)

        return theta * theta_sign

    def get_calibration_df(self):
        
        calibration_data = []

        for condition in self.calibration_conditions:
            PCA = self.get_calibration_pca(condition)
            pca_axes = PCA.components_
            pca_variance = PCA.explained_variance_ratio_
            mcp_ref_rotation = self.df[self.df['condition'] == condition]['trakstar_01'].iloc[0][:3, :3]
            pip_ref_rotation = self.df[self.df['condition'] == condition]['trakstar_02'].iloc[0][:3, :3]
            calibration_row = {'condition':condition, 
                            'mcp_ref_rotation': mcp_ref_rotation, 
                            'pip_ref_rotation': pip_ref_rotation, 
                            'PCA_axis1':pca_axes[0], 
                            'PCA_axis2':pca_axes[1], 
                            'PCA_axis3':pca_axes[2], 
                            'variance': pca_variance}

            calibration_data.append(calibration_row)
        
        calibration_df = pd.DataFrame(calibration_data)
        self.calibration_df = calibration_df

    def calculate_joint_angles(self, calibration_condition, condition):
        
        joint_axis = self.calibration_df[self.calibration_df['condition']==calibration_condition]['PCA_axis3'].iloc[0]
        if np.sign(joint_axis[2]) == 1:
            joint_axis = -joint_axis
        # pick the first transform of the dataset as the neutral pose
        mcp_ref_rotation = self.calibration_df[self.calibration_df['condition']==calibration_condition]['mcp_ref_rotation'].iloc[0]
        pip_ref_rotation = self.calibration_df[self.calibration_df['condition']==calibration_condition]['pip_ref_rotation'].iloc[0]
        # Rotate joint axis into frame
        mcp_axis = np.dot(np.linalg.inv(mcp_ref_rotation), joint_axis)
        pip_axis = np.dot(np.linalg.inv(pip_ref_rotation), joint_axis)
        print("Condition: %s"%condition)
        print("\nVariance ratio: %s"%self.calibration_df[self.calibration_df['condition']==calibration_condition]['variance'].iloc[0])
        print(joint_axis)
        print(mcp_axis)
        print(pip_axis)
        print("----------------------------\n")

        # Calculate the relative angle using the pre-defined reference transform and joint axis
        mcp_angle = [self.calculate_relative_angle(mcp_ref_rotation, t[:3, :3], mcp_axis) for t in self.df[self.df['condition'] == condition]['trakstar_01']]
        finger_angle = [self.calculate_relative_angle(pip_ref_rotation, t[:3, :3], pip_axis) for t in self.df[self.df['condition'] == condition]['trakstar_02']]
        # Find PIP angle using the difference 
        pip_angle = np.array(np.array([finger_angle]) - np.array([mcp_angle]))[0]
        indices = list(self.df[self.df['condition'] == condition].index)

        self.df.loc[indices, 'mcp_angle'] = mcp_angle
        self.df.loc[indices, 'pip_angle'] = pip_angle
        self.df.loc[indices, 'finger_angle'] = finger_angle

    def get_calibrated_joint_angles(self):

        for condition in self.conditions: 
            self.calculate_joint_angles(self.calibration_dict[condition], condition)

    def finalize_df(self):
        self.df.drop(columns=["trakstar0", "trakstar1", "trakstar2", "trakstar_01"], inplace = True)
        self.df = self.df[~self.df['futek'].isna()].copy()
        self.df = self.df[~self.df['motor_position'].isna()].copy()
        self.df.sort_values(by='raw_time', inplace=True)
        print(self.df.columns)
        self.df.reset_index(drop=True, inplace = True)
        self.df.reset_index(drop=False, inplace = True)
        print(self.df.columns)
        self.df.rename(columns= {'level_0' : 'index'}, inplace = True)
        self.df['time_unitless'] = np.arange(len(self.df))
        self.df.set_index(['condition', 'index'], inplace = True)
        self.df['c_index'] = np.zeros(len(self.df), dtype=int)
        for c in list(self.df.index.levels[0]):
            self.df.loc[(c, slice(None)), 'c_index'] = np.arange(0, len(self.df.loc[(c, slice(None)), 'c_index']), dtype=int)
        self.df.reset_index(inplace = True, drop = False)
        self.df.set_index(['condition', 'c_index', 'index'], inplace=True)
        self.df.rename(columns = {'level_0' : "time_unitless"}, inplace=True)
        self.df.index

    def get_mcp_pip_plot(self, condition):
        x = 'c_index'
        y = 'mcp_angle'
        z = 'pip_angle'
        df = self.df.loc[(condition, slice(None), slice(None)), :].copy().reset_index()
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                x_axis_label='time units',
                y_axis_label='angle (degrees)',
                title = condition + ' MCP and PIP angles',
                y_range = [-45, 200],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )

        p.circle(df[x], df[y], legend_label = y, alpha = 0.5)
        p.circle(df[x], df[z], color = 'red', legend_label = z, alpha = 0.5)

        return p

    def get_angle_plots(self):
        for i in self.conditions:
            p = self.get_mcp_pip_plot(i)
            p.title.text_font_size = '20pt'
            p.legend.label_text_font_size = '20pt'
            p.xaxis.axis_label_text_font_size = '20pt'
            p.yaxis.axis_label_text_font_size = '20pt'
            p.yaxis.major_label_text_font_size = '18pt'
            #p.legend.click_policy = 'hide'
            bokeh.io.show(p)
            self.angle_plots[i] = p

    def get_xy_plot(self, x, y, z=False):

        title = "%s vs. %s"%(y, x)
        if z != False:
            title = "%s and %s vs. %s"%(y, z, x)
        col = bokeh.palettes.d3['Category10'][10]
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=1000,
                height=600,
                x_axis_label=x,
                y_axis_label=y,
                title = title,
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )

        for i, c in enumerate(self.conditions):
            df = self.df.loc[(c, slice(None), slice(None)), :].copy().reset_index()
            p.circle(df[x], df[y], color = col[i], legend_label = c)
            if z != False: 
                p.circle(df[x], df[z], color = 'grey', legend_label = z)

        p.legend.location='top_left'
        p.legend.click_policy='hide'
        bokeh.io.show(p)
        return p
    
    def save_angle_plots(self):
        for p, name in self.angle_plots.items():
            export_png(p, filename="%s_angle_plot.png"%name)
        
def main(filename):
    csv_file = CSVParser()
    csv_file.file_to_df(filename)
    csv_file.parse_conditions()
    return csv_file



In [109]:
data.df.to_csv('p13_parsed_data.csv')

# Import data

In [106]:
#filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/demohand2/demohand1_compiled_data.csv'
#filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/test1_rosbags/tf_data.csv'
#filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/p1_jan_23_2024/p1_compiled_data.csv'
#filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/p3_rosbags/p3_compiled_data.csv'
filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/p13/p13_rosbags/p13_compiled_data.csv'
data = main(filename) 
print("Conditions: %s\nCalibration conditions: %s"%(data.conditions, data.calibration_conditions))


Conditions: {'passive_start', 'active_cube', 'active_rorcr', 'passive_end'}
Calibration conditions: ['passive_start_calibration', 'active_calibration', 'passive_end_calibration']


In [107]:
calibration_dict = {'passive_start': 'passive_start_calibration', 'active_rorcr': 'active_calibration', 'active_cube' : 'active_calibration', 'passive_end':'passive_end_calibration'}
data.calibration_dict = calibration_dict
data.get_calibration_df()
data.get_calibrated_joint_angles()
data.finalize_df()

Condition: passive_start

Variance ratio: [0.90435784 0.08328083 0.01236132]
[-0.44492686  0.08119419 -0.89187869]
[ 0.94803454 -0.22805288 -0.22186119]
[-0.2930614  -0.35048901 -0.88953497]
----------------------------

Condition: active_cube

Variance ratio: [0.85340362 0.12825646 0.01833992]
[-0.51432843  0.11802372 -0.84943314]
[ 0.82666026 -0.23857349 -0.50962291]
[-0.38739282 -0.34272416 -0.85584283]
----------------------------

Condition: active_rorcr

Variance ratio: [0.85340362 0.12825646 0.01833992]
[-0.51432843  0.11802372 -0.84943314]
[ 0.82666026 -0.23857349 -0.50962291]
[-0.38739282 -0.34272416 -0.85584283]
----------------------------

Condition: passive_end

Variance ratio: [0.82248742 0.15979511 0.01771746]
[-0.4667465   0.01945528 -0.88417713]
[ 0.94121156 -0.25099452 -0.22610296]
[-0.19720062 -0.36863843 -0.9084149 ]
----------------------------

Index(['condition', 'raw_time', 'time', 'futek', 'button', 'motor_position',
       'trakstar_02', 'mcp_angle', 'pip_angl

In [31]:
data.calibration_df.set_index(['condition'], inplace = True)

In [41]:
data.calibration_df.columns

Index(['mcp_ref_rotation', 'pip_ref_rotation', 'PCA_axis1', 'PCA_axis2',
       'PCA_axis3', 'variance'],
      dtype='object')

In [108]:
data.df

raw_time                    time  \
condition                 c_index index                                         
passive_start_calibration 0       0      1.706714e+09  2024-01-31 15:08:17.72   
                          1       1      1.706714e+09  2024-01-31 15:08:17.76   
                          2       2      1.706714e+09  2024-01-31 15:08:17.78   
                          3       3      1.706714e+09  2024-01-31 15:08:17.82   
                          4       4      1.706714e+09  2024-01-31 15:08:17.84   
...                                               ...                     ...   
passive_end               1807    30074  1.706716e+09  2024-01-31 15:48:29.96   
                          1808    30075  1.706716e+09  2024-01-31 15:48:30.03   
                          1809    30076  1.706716e+09  2024-01-31 15:48:30.07   
                          1810    30077  1.706716e+09  2024-01-31 15:48:30.09   
                          1811    30078  1.706716e+09  2024-01-31 15:48:30.13   

                                         futek button  motor_position  \
condition                 c_index index                                 
passive_start_calibration 0       0        0.0    NaN           0.000   
                          1       1        0.0    NaN           0.000   
                          2       2        0.0    NaN           0.000   
                          3       3        0.0    NaN           0.000   
                          4       4        0.0    NaN           0.000   
...                                        ...    ...             ...   
passive_end               1807    30074    0.0    NaN          -0.042   
                          1808    30075    0.0    NaN          -0.042   
                          1809    30076    0.0    NaN          -0.042   
                          1810    30077    0.0    NaN          -0.042   
                          1811    30078    0.0    NaN          -0.042   

                                                                               trakstar_02  \
condition                 c_index index                                                      
passive_start_calibration 0       0      [[0.23543548380113866, 0.9710664263371758, 0.0...   
                          1       1      [[0.23805716513238984, 0.9704254183594058, 0.0...   
                          2       2      [[0.2391370399502256, 0.9701456898201141, 0.04...   
                          3       3      [[0.24041984339716999, 0.9697683372762937, 0.0...   
                          4       4      [[0.24130399583786488, 0.9694966889995649, 0.0...   
...                                                                                    ...   
passive_end               1807    30074  [[-0.9663707917182374, 0.23173928436646862, -0...   
                          1808    30075  [[-0.9664094937289708, 0.2315015518009018, -0....   
                          1809    30076  [[-0.9664706375049724, 0.23136262045673853, -0...   
                          1810    30077  [[-0.9664903837355278, 0.23115594703675046, -0...   
                          1811    30078  [[-0.9664903837355278, 0.23115594703675046, -0...   

                                             mcp_angle      pip_angle  \
condition                 c_index index                                 
passive_start_calibration 0       0      7.521813e-316  5.547824e-316   
                          1       1      6.906050e-310  6.906050e-310   
                          2       2      6.906050e-310  6.906050e-310   
                          3       3      6.906050e-310  6.906050e-310   
                          4       4      6.906050e-310  6.906050e-310   
...                                                ...            ...   
passive_end               1807    30074   2.648452e+01   5.682314e+01   
                          1808    30075   2.649319e+01   5.682412e+01   
                          1809    30076   2.649653e+01   5.683374e+01   
               

In [100]:
data.get_angle_plots()

In [101]:
data.get_xy_plot('time_unitless', 'motor_position', 'futek')

figure(id='4a4572b6-65ed-4213-8dfc-bc6abac6ed27', ...)

In [75]:
data.df.index.sortorder

In [83]:
np.shape(data.df.index.values)

(32833,)

In [ ]:
data.get_xy_plot('', 'futek')

In [ ]:

x = 'motor_position'
y = 'futek'

hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=y,
        x_axis_label=x,
        title = '%s Force vs. Cable Retraction during Hand Opening'%plot_label[patient],
        #y_range = [-2, 20],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df1 = df[df['open'] > 0].copy()
df1.reset_index(inplace = True, drop = False)
df1.set_index(['condition', 'open'], inplace= True)
p.add_layout(Legend(), 'right')

for i, c in enumerate(conditions):
  num_opens = df1.loc[(c, slice(None))].index.values.max()

  for o in range(1, num_opens+1):
    df2 = df1.loc[(c, o)].copy()
    p.line(df2[x], df2[y], color = col[i], legend_label = c, line_width = 3, alpha = 0.5)

p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='bottom'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [ ]:
dff = p13_data.df.loc[('passive_start_calibration', slice(None), slice(None)), :].reset_index()
dff.head(3)

# old